# ✈️ Daily Nigerian Domestic Flight Price Alert

**Author:** ijosh  
**Week:** 8 — Community Contribution  
**Description:** Autonomous Multi-Agent system that scans Nigerian domestic routes for live fares via the Amadeus API, uses a 6-LLM ensemble to recommend the best flight, and sends Pushover push notifications.

## Features
- 🌐 **Live Flight Data**: Amadeus API (free sandbox) for real Nigerian routes
- 🤖 **6-LLM Ensemble**: GPT-4o-mini · Claude Haiku · Gemini Flash + Llama 3.1 · Mixtral · Gemma-2 (via Groq)
- 🛫 **Nigerian Routes**: LOS ↔ ABV ↔ PHC ↔ KAN ↔ ENO
- 🔔 **Pushover Notifications**: Instant push alerts to your phone
- 🌐 **Gradio UI**: 4-tab polished dark interface
- ☁️ **Modal Deployment**: Daily scheduled scans in production



In [ ]:
import os
import json
import time
import random
import logging
import requests
from datetime import datetime, timedelta
from typing import List, Optional, Dict, Any
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from openai import OpenAI
import anthropic
import google.generativeai as genai
from groq import Groq
import gradio as gr

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
print('✅ Imports loaded')

In [ ]:
# ── Configuration & Environment ──────────────────────────────────────────────
load_dotenv(override=True)

# API keys (never hardcoded)
OPENAI_API_KEY       = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY    = os.getenv('ANTHROPIC_API_KEY')
GOOGLE_API_KEY       = os.getenv('GOOGLE_API_KEY')
GROQ_API_KEY         = os.getenv('GROQ_API_KEY')
AMADEUS_CLIENT_ID    = os.getenv('AMADEUS_CLIENT_ID')
AMADEUS_CLIENT_SECRET = os.getenv('AMADEUS_CLIENT_SECRET')
PUSHOVER_USER        = os.getenv('PUSHOVER_USER')
PUSHOVER_TOKEN       = os.getenv('PUSHOVER_TOKEN')

# Nigerian airports
AIRPORTS = {
    'LOS': 'Lagos (Murtala Muhammed)',
    'ABV': 'Abuja (Nnamdi Azikiwe)',
    'PHC': 'Port Harcourt International',
    'KAN': 'Kano (Mallam Aminu)',
    'ENO': 'Enugu (Akanu Ibiam)',
}

ROUTES = [
    ('LOS', 'ABV'), ('ABV', 'LOS'),
    ('LOS', 'PHC'), ('PHC', 'LOS'),
    ('LOS', 'KAN'), ('KAN', 'LOS'),
    ('LOS', 'ENO'), ('ENO', 'LOS'),
    ('ABV', 'PHC'), ('PHC', 'ABV'),
]

# Nigerian carriers with realistic price ranges (NGN)
NIGERIAN_AIRLINES = [
    {'name': 'Air Peace',      'code': 'P4', 'iata': 'P4'},
    {'name': 'Arik Air',       'code': 'W3', 'iata': 'W3'},
    {'name': 'Ibom Air',       'code': 'QI', 'iata': 'QI'},
    {'name': 'United Nigeria', 'code': 'UN', 'iata': 'UN'},
]

PRICE_RANGES_NGN = {
    'LOS-ABV': (28000,  85000),
    'ABV-LOS': (28000,  85000),
    'LOS-PHC': (32000,  95000),
    'PHC-LOS': (32000,  95000),
    'LOS-KAN': (38000, 110000),
    'KAN-LOS': (38000, 110000),
    'LOS-ENO': (30000,  88000),
    'ENO-LOS': (30000,  88000),
    'ABV-PHC': (33000,  98000),
    'PHC-ABV': (33000,  98000),
}

# Check key availability
for var, label in [
    ('OPENAI_API_KEY', 'OpenAI'), ('ANTHROPIC_API_KEY', 'Anthropic'),
    ('GOOGLE_API_KEY', 'Google'), ('GROQ_API_KEY', 'Groq'),
    ('AMADEUS_CLIENT_ID', 'Amadeus ID'), ('AMADEUS_CLIENT_SECRET', 'Amadeus Secret'),
    ('PUSHOVER_USER', 'Pushover User'), ('PUSHOVER_TOKEN', 'Pushover Token'),
]:
    val = os.getenv(var)
    icon = '✅' if val else '⚠️'
    print(f'{icon} {label}: {"Set" if val else "Not set (will use fallback)"}')

In [ ]:
# ── Pydantic Models ───────────────────────────────────────────────────────────

class FlightOption(BaseModel):
    """A single available flight option."""
    airline: str
    flight_number: str
    origin: str
    destination: str
    departure_time: str
    arrival_time: str
    price_ngn: float
    currency: str = 'NGN'
    seats_available: Optional[int] = None
    source: str = 'live'  # 'live' or 'mock'


class FlightSearchResult(BaseModel):
    """Full results from a flight search for a route."""
    route: str
    search_date: str
    cheapest_flight: FlightOption
    all_options: List[FlightOption]
    summary: str


class AlertNotification(BaseModel):
    """Pushover push notification payload."""
    title: str
    message: str
    priority: int = Field(default=0, ge=-2, le=2)
    url: Optional[str] = None


class EnsembleVote(BaseModel):
    """One LLM's vote on the best flight."""
    model_name: str
    recommended_flight_number: str
    reasoning: str
    confidence: float = Field(default=0.5, ge=0.0, le=1.0)


class AgentState(BaseModel):
    """Mutable state maintained by the orchestrator across tool calls."""
    origin: str
    destination: str
    travel_date: str
    search_result: Optional[FlightSearchResult] = None
    ensemble_votes: List[EnsembleVote] = []
    final_recommendation: Optional[FlightOption] = None
    notification_sent: bool = False
    iteration: int = 0
    logs: List[str] = []


print('✅ Pydantic models defined')

In [ ]:
# ── Live Flight Search via Amadeus API ────────────────────────────────────────

# USD → NGN approximate exchange rate (Amadeus returns NGN if requested)
USD_TO_NGN = 1600

_amadeus_token: Optional[str] = None
_amadeus_token_expires: float = 0


def _get_amadeus_token() -> Optional[str]:
    """Obtain or refresh an Amadeus OAuth2 bearer token."""
    global _amadeus_token, _amadeus_token_expires
    if not AMADEUS_CLIENT_ID or not AMADEUS_CLIENT_SECRET:
        return None
    if _amadeus_token and time.time() < _amadeus_token_expires:
        return _amadeus_token
    try:
        resp = requests.post(
            'https://test.api.amadeus.com/v1/security/oauth2/token',
            data={
                'grant_type': 'client_credentials',
                'client_id': AMADEUS_CLIENT_ID,
                'client_secret': AMADEUS_CLIENT_SECRET,
            },
            timeout=10,
        )
        data = resp.json()
        _amadeus_token = data['access_token']
        _amadeus_token_expires = time.time() + data.get('expires_in', 1799) - 60
        logger.info('✅ Amadeus token obtained')
        return _amadeus_token
    except Exception as e:
        logger.warning(f'⚠️ Amadeus token error: {e}')
        return None


def _airline_name_from_code(code: str) -> str:
    """Map IATA airline code to carrier name (Nigerian carriers + fallback)."""
    CARRIER_MAP = {
        'P4': 'Air Peace', 'W3': 'Arik Air', 'QI': 'Ibom Air',
        'UN': 'United Nigeria', 'WT': 'Overland Airways',
        'VK': 'ValueJet', '9J': 'Dana Air',
    }
    return CARRIER_MAP.get(code, f'Airline {code}')


def _parse_amadeus_offers(data: Dict, origin: str, destination: str) -> List[FlightOption]:
    """Parse raw Amadeus flight-offers response into FlightOption list."""
    options: List[FlightOption] = []
    for offer in data.get('data', []):
        try:
            seg = offer['itineraries'][0]['segments'][0]
            price_raw = float(offer['price']['total'])
            currency = offer['price']['currency']
            # Convert to NGN if price comes back in USD/other
            price_ngn = price_raw if currency == 'NGN' else price_raw * USD_TO_NGN
            carrier_code = seg['carrierCode']
            dep_dt = seg['departure']['at']  # ISO datetime string
            arr_dt = seg['arrival']['at']
            dep_time = dep_dt[11:16]   # HH:MM
            arr_time = arr_dt[11:16]
            seats = offer.get('numberOfBookableSeats')
            fn = f"{carrier_code}{seg['number']}"
            options.append(FlightOption(
                airline=_airline_name_from_code(carrier_code),
                flight_number=fn,
                origin=origin,
                destination=destination,
                departure_time=dep_time,
                arrival_time=arr_time,
                price_ngn=round(price_ngn, 0),
                seats_available=int(seats) if seats else None,
                source='live',
            ))
        except (KeyError, IndexError, ValueError) as e:
            logger.debug(f'Skipping offer parse error: {e}')
    return options


def _live_search(origin: str, destination: str, date: str) -> List[FlightOption]:
    """Fetch live flights from Amadeus API. Returns empty list on failure."""
    token = _get_amadeus_token()
    if not token:
        return []
    for attempt in range(3):
        try:
            resp = requests.get(
                'https://test.api.amadeus.com/v2/shopping/flight-offers',
                headers={'Authorization': f'Bearer {token}'},
                params={
                    'originLocationCode': origin,
                    'destinationLocationCode': destination,
                    'departureDate': date,
                    'adults': 1,
                    'currencyCode': 'NGN',
                    'max': 20,
                    'nonStop': 'true',
                },
                timeout=15,
            )
            if resp.status_code == 200:
                options = _parse_amadeus_offers(resp.json(), origin, destination)
                logger.info(f'📡 Amadeus: {len(options)} live flights for {origin}-{destination}')
                return options
            elif resp.status_code == 429:
                wait = 2 ** attempt
                logger.warning(f'⏳ Amadeus rate limit — waiting {wait}s')
                time.sleep(wait)
            else:
                logger.warning(f'⚠️ Amadeus HTTP {resp.status_code}: {resp.text[:200]}')
                return []
        except requests.RequestException as e:
            logger.warning(f'⚠️ Amadeus request error (attempt {attempt+1}): {e}')
            time.sleep(2 ** attempt)
    return []


def _mock_search(origin: str, destination: str, date: str) -> List[FlightOption]:
    """Generate realistic Nigerian domestic flight options when live API is unavailable."""
    route_key = f'{origin}-{destination}'
    price_min, price_max = PRICE_RANGES_NGN.get(route_key, (30000, 120000))
    rng = random.Random(hash(route_key + date))
    options: List[FlightOption] = []
    for airline in NIGERIAN_AIRLINES:
        for _ in range(rng.randint(1, 2)):
            dep_h = rng.randint(6, 20)
            dep_m = rng.choice([0, 15, 30, 45])
            dur   = rng.randint(50, 95)
            arr_h = (dep_h * 60 + dep_m + dur) // 60
            arr_m = (dep_h * 60 + dep_m + dur) % 60
            price = round(rng.uniform(price_min, price_max) / 1000) * 1000
            fn    = f"{airline['code']}{rng.randint(100, 999)}"
            options.append(FlightOption(
                airline=airline['name'],
                flight_number=fn,
                origin=origin,
                destination=destination,
                departure_time=f'{dep_h:02d}:{dep_m:02d}',
                arrival_time=f'{arr_h % 24:02d}:{arr_m:02d}',
                price_ngn=float(price),
                seats_available=rng.randint(3, 48),
                source='mock',
            ))
    options.sort(key=lambda x: x.price_ngn)
    return options


def search_flights(origin: str, destination: str, date: str) -> FlightSearchResult:
    """
    Search for available flights on a Nigerian domestic route.

    Tries Amadeus live API first; falls back to Nigerian market simulation.

    Args:
        origin:      IATA code of origin airport (e.g. 'LOS')
        destination: IATA code of destination airport (e.g. 'ABV')
        date:        Travel date in YYYY-MM-DD format

    Returns:
        FlightSearchResult with all options and cheapest flight
    """
    logger.info(f'🔍 Searching {origin} → {destination} on {date}')
    options = _live_search(origin, destination, date)
    data_source = 'live (Amadeus)'
    if not options:
        options = _mock_search(origin, destination, date)
        data_source = 'simulated (Nigerian market)'
        logger.info(f'📊 Using {data_source} data')

    options.sort(key=lambda x: x.price_ngn)
    cheapest = options[0]
    route_key = f'{origin}-{destination}'
    return FlightSearchResult(
        route=route_key,
        search_date=date,
        cheapest_flight=cheapest,
        all_options=options,
        summary=(
            f'{len(options)} flights found ({data_source}) from '
            f'{AIRPORTS.get(origin, origin)} to {AIRPORTS.get(destination, destination)} '
            f'on {date}. Cheapest: {cheapest.airline} at ₦{cheapest.price_ngn:,.0f} '
            f'(Flight {cheapest.flight_number}, departs {cheapest.departure_time})'
        ),
    )


print('✅ Flight search tools ready')
print(f'   Amadeus live data: {"enabled" if AMADEUS_CLIENT_ID else "disabled (add AMADEUS_CLIENT_ID/SECRET to .env)"}')

In [ ]:
# ── Pushover Notification Tool ────────────────────────────────────────────────

def send_pushover_alert(notification: AlertNotification) -> Dict[str, Any]:
    """
    Send a push notification via Pushover with retry logic.

    Args:
        notification: AlertNotification with title, message, priority and optional URL

    Returns:
        dict with 'success' bool and 'response' or 'error'
    """
    user_key  = os.getenv('PUSHOVER_USER')
    api_token = os.getenv('PUSHOVER_TOKEN')
    if not user_key or not api_token:
        msg = 'Pushover credentials not set — skipping notification'
        logger.warning(f'⚠️ {msg}')
        return {'success': False, 'error': msg}

    payload: Dict[str, Any] = {
        'token':   api_token,
        'user':    user_key,
        'title':   notification.title,
        'message': notification.message,
        'priority': notification.priority,
        'sound':   'cashregister',
    }
    if notification.url:
        payload['url'] = notification.url
        payload['url_title'] = 'Book Now'

    for attempt in range(3):
        try:
            resp = requests.post(
                'https://api.pushover.net/1/messages.json',
                data=payload, timeout=10,
            )
            if resp.status_code == 200:
                logger.info(f'🔔 Pushover sent: {notification.title}')
                return {'success': True, 'response': resp.json()}
            logger.warning(f'⚠️ Pushover attempt {attempt+1}: HTTP {resp.status_code}')
        except requests.RequestException as e:
            logger.warning(f'⚠️ Pushover attempt {attempt+1}: {e}')
        if attempt < 2:
            time.sleep(2 ** attempt)

    return {'success': False, 'error': 'Max retries reached'}


print('✅ Pushover notification tool ready')

In [ ]:
# ── LLM Client Setup (6 models) ───────────────────────────────────────────────

openai_client    = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))
groq_client      = Groq(api_key=os.getenv('GROQ_API_KEY'))


def _call_openai(prompt: str, model: str = 'gpt-4o-mini') -> str:
    """Query OpenAI GPT-4o-mini and return text response."""
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300, temperature=0.3,
    )
    return resp.choices[0].message.content.strip()


def _call_anthropic(prompt: str) -> str:
    """Query Anthropic Claude Haiku and return text response."""
    msg = anthropic_client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=300,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return msg.content[0].text.strip()


def _call_google(prompt: str) -> str:
    """Query Google Gemini 1.5 Flash and return text response."""
    model = genai.GenerativeModel('gemini-1.5-flash')
    return model.generate_content(prompt).text.strip()


def _call_groq(prompt: str, model: str) -> str:
    """Query a Groq-hosted open-source model and return text response."""
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300, temperature=0.3,
    )
    return resp.choices[0].message.content.strip()


# Model registry: display_name → callable
LLM_REGISTRY = {
    'GPT-4o-mini':   lambda p: _call_openai(p, 'gpt-4o-mini'),
    'Claude-Haiku':  _call_anthropic,
    'Gemini-Flash':  _call_google,
    'Llama-3.1-8B':  lambda p: _call_groq(p, 'llama-3.1-8b-instant'),
    'Mixtral-8x7B':  lambda p: _call_groq(p, 'mixtral-8x7b-32768'),
    'Gemma-2-9B':    lambda p: _call_groq(p, 'gemma2-9b-it'),
}

print(f'✅ {len(LLM_REGISTRY)} LLM clients configured: {", ".join(LLM_REGISTRY)}')

In [ ]:
# ── Ensemble Agent ────────────────────────────────────────────────────────────

def _build_vote_prompt(result: FlightSearchResult) -> str:
    """Build the flight recommendation prompt for LLM ensemble voting."""
    lines = [
        'You are a flight recommendation expert for Nigerian domestic routes.',
        f'Analyze these flights ({result.route}) on {result.search_date}:',
        '',
    ]
    for i, f in enumerate(result.all_options, 1):
        seat_str = str(f.seats_available) if f.seats_available else 'N/A'
        lines.append(
            f'{i}. [{f.flight_number}] {f.airline} | '
            f'NGN {f.price_ngn:,.0f} | '
            f'Dep {f.departure_time} → Arr {f.arrival_time} | '
            f'Seats: {seat_str}'
        )
    lines += [
        '',
        'Pick the BEST VALUE flight (balance price, availability, and timing).',
        'Respond ONLY with valid JSON:',
        '{"flight_number": "XXNNN", "reasoning": "brief reason", "confidence": 0.85}',
    ]
    return '\n'.join(lines)


def _parse_vote(raw: str, model_name: str, fallback: FlightOption) -> EnsembleVote:
    """Parse an LLM vote response; falls back to cheapest if JSON parsing fails."""
    try:
        cleaned = raw.strip().lstrip('```json').lstrip('```').rstrip('```').strip()
        # Extract first JSON object
        start, end = cleaned.index('{'), cleaned.rindex('}') + 1
        data = json.loads(cleaned[start:end])
        return EnsembleVote(
            model_name=model_name,
            recommended_flight_number=data.get('flight_number', fallback.flight_number),
            reasoning=data.get('reasoning', 'No reasoning provided'),
            confidence=float(data.get('confidence', 0.5)),
        )
    except Exception:
        return EnsembleVote(
            model_name=model_name,
            recommended_flight_number=fallback.flight_number,
            reasoning='Defaulted to cheapest (JSON parse failed)',
            confidence=0.2,
        )


def ensemble_vote(
    search_result: FlightSearchResult,
    selected_models: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """
    Run 6-LLM ensemble voting to determine the best flight.

    Each LLM independently evaluates all options and votes for the best value.
    Votes are weighted by confidence and tallied for a final recommendation.

    Args:
        search_result:   FlightSearchResult to evaluate
        selected_models: Subset of LLM_REGISTRY keys to use (default: all)

    Returns:
        dict with 'votes' list, 'winner' FlightOption, 'vote_counts' dict
    """
    prompt   = _build_vote_prompt(search_result)
    cheapest = search_result.cheapest_flight
    models   = {k: v for k, v in LLM_REGISTRY.items()
                if not selected_models or k in selected_models}

    votes: List[EnsembleVote] = []
    for name, query_fn in models.items():
        try:
            raw  = query_fn(prompt)
            vote = _parse_vote(raw, name, cheapest)
        except Exception as e:
            logger.warning(f'⚠️ {name} vote failed: {e}')
            vote = EnsembleVote(
                model_name=name,
                recommended_flight_number=cheapest.flight_number,
                reasoning=f'Error: {str(e)[:80]}',
                confidence=0.0,
            )
        votes.append(vote)
        logger.info(f'🗳️ {name} → {vote.recommended_flight_number} (conf={vote.confidence:.2f})')

    # Tally with confidence weighting
    scores: Dict[str, float] = {}
    counts: Dict[str, int]   = {}
    for v in votes:
        fn = v.recommended_flight_number
        scores[fn] = scores.get(fn, 0.0) + v.confidence
        counts[fn] = counts.get(fn, 0)   + 1

    winner_fn = max(scores, key=scores.get) if scores else cheapest.flight_number
    winner    = next(
        (f for f in search_result.all_options if f.flight_number == winner_fn),
        cheapest,
    )
    return {'votes': votes, 'winner': winner, 'vote_counts': counts, 'vote_scores': scores}


print('✅ Ensemble agent ready')

In [ ]:
# ── Orchestrator / Agent Loop ─────────────────────────────────────────────────

TOOL_SPECS = [
    {
        'type': 'function',
        'function': {
            'name': 'search_flights',
            'description': 'Search live Nigerian domestic flight offers for a route and date',
            'parameters': {
                'type': 'object',
                'properties': {
                    'origin':      {'type': 'string', 'description': 'IATA origin code'},
                    'destination': {'type': 'string', 'description': 'IATA destination code'},
                    'date':        {'type': 'string', 'description': 'Date YYYY-MM-DD'},
                },
                'required': ['origin', 'destination', 'date'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'ensemble_vote',
            'description': 'Run 6-LLM ensemble to pick the best flight from current search results',
            'parameters': {
                'type': 'object',
                'properties': {
                    'note': {'type': 'string', 'description': 'Optional note for logging'},
                },
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'send_pushover_alert',
            'description': 'Send a push notification with the best flight details',
            'parameters': {
                'type': 'object',
                'properties': {
                    'title':    {'type': 'string'},
                    'message':  {'type': 'string'},
                    'priority': {'type': 'integer', 'minimum': -2, 'maximum': 2},
                },
                'required': ['title', 'message', 'priority'],
            },
        },
    },
]


class FlightOrchestrator:
    """
    Autonomous orchestrator that coordinates flight search, ensemble voting,
    and push notifications via an OpenAI function-calling agent loop.
    """

    def __init__(self, max_iterations: int = 10) -> None:
        """Initialise orchestrator with configurable iteration limit."""
        self.max_iterations       = max_iterations
        self.state: Optional[AgentState] = None
        self.messages: List[Dict] = []
        self.notification_history: List[Dict] = []

    def _dispatch_tool(self, name: str, args: Dict, state: AgentState) -> str:
        """Route a tool call to the correct handler and update agent state."""
        if name == 'search_flights':
            result = search_flights(
                origin=args['origin'],
                destination=args['destination'],
                date=args['date'],
            )
            state.search_result = result
            state.logs.append(f'🔍 Found {len(result.all_options)} flights on {result.route}')
            return result.model_dump_json()

        elif name == 'ensemble_vote':
            if not state.search_result:
                return json.dumps({'error': 'Call search_flights first'})
            ev = ensemble_vote(state.search_result)
            state.ensemble_votes      = ev['votes']
            state.final_recommendation = ev['winner']
            winner = ev['winner']
            state.logs.append(
                f'🏆 Ensemble picked {winner.airline} {winner.flight_number} '
                f'@ ₦{winner.price_ngn:,.0f}'
            )
            return json.dumps({'winner': winner.model_dump(), 'vote_counts': ev['vote_counts']})

        elif name == 'send_pushover_alert':
            notif = AlertNotification(
                title=args['title'],
                message=args['message'],
                priority=args.get('priority', 0),
            )
            result = send_pushover_alert(notif)
            if result['success']:
                state.notification_sent = True
                rec = {
                    'timestamp': datetime.now().isoformat(),
                    'route':     f"{state.origin}-{state.destination}",
                    'title':     notif.title,
                    'message':   notif.message,
                    'price':     state.final_recommendation.price_ngn
                                 if state.final_recommendation else 0,
                }
                self.notification_history.append(rec)
                state.logs.append(f'🔔 Alert sent: {notif.title}')
            return json.dumps(result)

        return json.dumps({'error': f'Unknown tool: {name}'})

    def run(
        self,
        origin: str,
        destination: str,
        travel_date: str,
        price_threshold: Optional[float] = None,
        selected_models: Optional[List[str]] = None,
        log_callback: Optional[Any] = None,
    ) -> AgentState:
        """
        Run the full autonomous agent loop for a flight price alert.

        Sequence: search_flights → ensemble_vote → send_pushover_alert → done

        Args:
            origin:          IATA origin code
            destination:     IATA destination code
            travel_date:     Date string YYYY-MM-DD
            price_threshold: Optional NGN threshold for alert
            selected_models: Subset of LLM names for ensemble
            log_callback:    Optional callable(str) for live UI streaming

        Returns:
            Completed AgentState
        """
        state = AgentState(origin=origin, destination=destination, travel_date=travel_date)
        self.state = state

        threshold_str = f' (alert if ≤ ₦{price_threshold:,.0f})' if price_threshold else ''
        sys_prompt = (
            f'You are an autonomous Nigerian domestic flight price alert agent.\n'
            f'Task: find the best flight {origin} → {destination} on {travel_date}{threshold_str}.\n'
            f'Steps:\n'
            f'1. Call search_flights to get live prices\n'
            f'2. Call ensemble_vote for AI consensus\n'
            f'3. Call send_pushover_alert with a concise alert message\n'
            f'4. Summarise and stop. Be decisive — call each tool exactly once.'
        )
        self.messages = [
            {'role': 'system', 'content': sys_prompt},
            {'role': 'user',   'content': f'Find the best flight {origin}→{destination} on {travel_date} and notify me.'},
        ]

        def log(msg: str) -> None:
            state.logs.append(msg)
            logger.info(msg)
            if log_callback:
                log_callback(msg)

        log(f'🚀 Agent started: {origin} → {destination} on {travel_date}')

        for iteration in range(self.max_iterations):
            state.iteration = iteration + 1
            try:
                resp = openai_client.chat.completions.create(
                    model='gpt-4o-mini',
                    messages=self.messages,
                    tools=TOOL_SPECS,
                    tool_choice='auto',
                    max_tokens=1024,
                )
            except Exception as e:
                log(f'❌ Orchestrator API error: {e}')
                break

            choice = resp.choices[0]
            self.messages.append(choice.message)

            if choice.finish_reason == 'stop':
                log(f'✅ Agent done after {iteration+1} iteration(s)')
                if choice.message.content:
                    log(f'📋 {choice.message.content}')
                break

            if choice.finish_reason == 'tool_calls' and choice.message.tool_calls:
                for tc in choice.message.tool_calls:
                    args = json.loads(tc.function.arguments or '{}')
                    log(f'🔧 Tool: {tc.function.name}({list(args.keys())})')
                    result = self._dispatch_tool(tc.function.name, args, state)
                    self.messages.append({
                        'role': 'tool',
                        'content': result,
                        'tool_call_id': tc.id,
                    })

        return state


orchestrator = FlightOrchestrator(max_iterations=10)
print('✅ Orchestrator ready')

In [ ]:
# ── Gradio UI ─────────────────────────────────────────────────────────────────

AIRPORT_OPTIONS = [f'{code} - {name}' for code, name in AIRPORTS.items()]
MODEL_OPTIONS   = list(LLM_REGISTRY.keys())
_alert_config: Dict[str, Any] = {'active': False, 'threshold': 50000}

DARK_CSS = """
.gradio-container { background:#0b0f1e !important; color:#dde6f0 !important; }
.tab-nav button { background:#162040 !important; color:#7ab3ff !important;
                  border:1px solid #2a3f6e !important; border-radius:8px !important; font-weight:600; }
.tab-nav button.selected { background:#0052cc !important; color:#fff !important; }
.winner-card { background:linear-gradient(135deg,#162040,#0b1426);
               border:2px solid #00b368; border-radius:12px;
               padding:18px; margin:8px 0; }
.alert-on  { background:#002a10; border:1px solid #00d67a; border-radius:8px;
             padding:8px 16px; color:#00e87a; font-weight:700; }
.alert-off { color:#666; padding:8px; }
"""


def _code(airport_str: str) -> str:
    """Extract 3-letter IATA code from a 'CODE - Name' dropdown string."""
    return airport_str.split(' - ')[0]


def ui_search(
    origin_str: str, dest_str: str, date_str: str, models: List[str],
    progress=gr.Progress(),
) -> tuple:
    """Handle Tab 1 search: run agent loop and return formatted results."""
    origin, dest = _code(origin_str), _code(dest_str)
    if origin == dest:
        return '<p style="color:red">⚠️ Origin and destination must differ.</p>', '', []

    logs: List[str] = []
    progress(0.1, desc='🔍 Searching live prices...')

    state = orchestrator.run(
        origin=origin, destination=dest, travel_date=date_str,
        selected_models=models or None,
        log_callback=logs.append,
    )
    progress(0.9, desc='🏆 Finalising...')

    if not state.search_result:
        return '<p style="color:red">❌ No flights found.</p>', '\n'.join(logs), []

    winner = state.final_recommendation or state.search_result.cheapest_flight
    src_badge = '📡 Live' if winner.source == 'live' else '📊 Simulated'
    card = (
        f'<div class="winner-card">'
        f'<h2>🏆 Best Flight — {winner.airline} {src_badge}</h2>'
        f'<p>✈️ <b>Flight:</b> {winner.flight_number} &nbsp;|&nbsp;'
        f' 🛫 <b>Route:</b> {winner.origin} → {winner.destination}</p>'
        f'<p>🕐 <b>Dep:</b> {winner.departure_time} &nbsp;|&nbsp;'
        f' 🕐 <b>Arr:</b> {winner.arrival_time}</p>'
        f'<p>💰 <b>Price:</b> ₦{winner.price_ngn:,.0f} &nbsp;|&nbsp;'
        f' 💺 <b>Seats:</b> {winner.seats_available or "N/A"}</p>'
        f'</div>'
    )
    rows = [
        [f.airline, f.flight_number, f.departure_time, f.arrival_time,
         f'₦{f.price_ngn:,.0f}', str(f.seats_available or 'N/A'), f.source]
        for f in state.search_result.all_options
    ]
    return card, '\n'.join(logs), rows


def ui_set_alert(
    origin_str: str, dest_str: str, date_str: str, threshold: float, enabled: bool,
) -> str:
    """Handle Tab 2: save alert config and return status badge HTML."""
    _alert_config.update({
        'active': enabled, 'threshold': threshold,
        'origin': _code(origin_str), 'destination': _code(dest_str), 'date': date_str,
    })
    if enabled:
        return (
            f'<div class="alert-on">'
            f'🔔 ALERT ACTIVE: {_code(origin_str)} → {_code(dest_str)} '
            f'below ₦{threshold:,.0f} on {date_str}'
            f'</div>'
        )
    return '<div class="alert-off">⭕ Alert inactive</div>'


def ui_run_console(origin_str: str, dest_str: str, date_str: str) -> tuple:
    """Handle Tab 3: run agent and stream logs + vote table."""
    origin, dest = _code(origin_str), _code(dest_str)
    state = orchestrator.run(origin=origin, destination=dest, travel_date=date_str)
    log_text = '\n'.join(state.logs)
    vote_rows = [
        [v.model_name, v.recommended_flight_number,
         f'{v.confidence:.0%}', v.reasoning[:90]]
        for v in state.ensemble_votes
    ]
    return log_text, vote_rows


def ui_history() -> List[List[str]]:
    """Return notification history as table rows for Tab 4."""
    if not orchestrator.notification_history:
        return [['—', '—', '—', '—']]
    return [
        [r['timestamp'][:19], r['route'], f'₦{r["price"]:,.0f}', r['title']]
        for r in orchestrator.notification_history
    ]


with gr.Blocks(title='🛫 Nigerian Flight Price Alert', css=DARK_CSS,
               theme=gr.themes.Base(primary_hue='green')) as demo:

    gr.Markdown(
        '# 🛫 Nigerian Domestic Flight Price Alert\n'
        '### 6-LLM Ensemble AI · Air Peace · Arik · Ibom Air · United Nigeria\n---'
    )

    with gr.Tabs():

        # ── Tab 1: Search ────────────────────────────────────────────────────
        with gr.Tab('✈️ Search Flights'):
            with gr.Row():
                with gr.Column(scale=1):
                    t1_origin = gr.Dropdown(AIRPORT_OPTIONS, value='LOS - Lagos (Murtala Muhammed)', label='🛫 Origin')
                    t1_dest   = gr.Dropdown(AIRPORT_OPTIONS, value='ABV - Abuja (Nnamdi Azikiwe)', label='🛬 Destination')
                    t1_date   = gr.Textbox(
                        label='📅 Travel Date (YYYY-MM-DD)',
                        value=(datetime.now() + timedelta(days=3)).strftime('%Y-%m-%d'),
                    )
                    t1_models = gr.CheckboxGroup(MODEL_OPTIONS, value=MODEL_OPTIONS, label='🤖 LLM Ensemble')
                    t1_btn    = gr.Button('🔍 Find Cheapest Flight', variant='primary', size='lg')
                with gr.Column(scale=2):
                    t1_card   = gr.HTML(label='🏆 Best Flight')
                    t1_log    = gr.Textbox(label='📝 Agent Log', lines=5, interactive=False)
                    t1_table  = gr.Dataframe(
                        headers=['Airline', 'Flight', 'Departs', 'Arrives', 'Price (₦)', 'Seats', 'Source'],
                        label='📋 All Flights', wrap=True,
                    )
            t1_btn.click(ui_search, [t1_origin, t1_dest, t1_date, t1_models], [t1_card, t1_log, t1_table])

        # ── Tab 2: Price Alert ───────────────────────────────────────────────
        with gr.Tab('🔔 Set Price Alert'):
            with gr.Row():
                with gr.Column():
                    t2_origin    = gr.Dropdown(AIRPORT_OPTIONS, value='LOS - Lagos (Murtala Muhammed)', label='🛫 Origin')
                    t2_dest      = gr.Dropdown(AIRPORT_OPTIONS, value='PHC - Port Harcourt International', label='🛬 Destination')
                    t2_date      = gr.Textbox(
                        label='📅 Date', value=(datetime.now() + timedelta(days=7)).strftime('%Y-%m-%d'),
                    )
                    t2_threshold = gr.Number(label='💰 Price Threshold (₦)', value=50000, minimum=5000, step=5000)
                    t2_toggle    = gr.Checkbox(label='Enable Daily Monitoring', value=False)
                    t2_btn       = gr.Button('🔔 Set Alert', variant='primary')
                with gr.Column():
                    t2_status = gr.HTML('<div class="alert-off">⭕ Alert inactive</div>')
                    gr.Markdown(
                        '### How It Works\n'
                        '- System checks your route daily at 7AM WAT\n'
                        '- When price drops below threshold → Pushover alert\n'
                        '- Deploy to Modal for fully automated production monitoring\n\n'
                        '📱 Install **Pushover** on your phone to receive instant alerts!'
                    )
            t2_btn.click(ui_set_alert, [t2_origin, t2_dest, t2_date, t2_threshold, t2_toggle], [t2_status])

        # ── Tab 3: Agent Console ─────────────────────────────────────────────
        with gr.Tab('🤖 Agent Console'):
            with gr.Row():
                t3_origin = gr.Dropdown(AIRPORT_OPTIONS, value='ABV - Abuja (Nnamdi Azikiwe)', label='🛫 Origin')
                t3_dest   = gr.Dropdown(AIRPORT_OPTIONS, value='LOS - Lagos (Murtala Muhammed)', label='🛬 Destination')
                t3_date   = gr.Textbox(
                    label='📅 Date', value=(datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d'),
                )
                t3_btn    = gr.Button('▶️ Run Agent Loop', variant='primary')
            with gr.Row():
                t3_log   = gr.Textbox(label='📝 Agent Reasoning Log', lines=14, interactive=False)
                t3_votes = gr.Dataframe(
                    headers=['Model', 'Voted Flight', 'Confidence', 'Reasoning'],
                    label='🗳️ Ensemble Votes', wrap=True,
                )
            t3_btn.click(ui_run_console, [t3_origin, t3_dest, t3_date], [t3_log, t3_votes])

        # ── Tab 4: Notification History ──────────────────────────────────────
        with gr.Tab('📊 Notification History'):
            t4_refresh = gr.Button('🔄 Refresh')
            t4_table   = gr.Dataframe(
                headers=['Timestamp', 'Route', 'Price (₦)', 'Alert Title'],
                label='📬 Past Pushover Alerts', wrap=True,
            )
            t4_refresh.click(ui_history, outputs=[t4_table])
            demo.load(ui_history, outputs=[t4_table])

    gr.Markdown(
        '---\n*Built with ❤️ for Nigerian travelers — '
        'GPT-4o-mini · Claude Haiku · Gemini Flash · Llama 3.1 · Mixtral · Gemma-2*'
    )

demo.launch(inbrowser=True)  # Uncomment to launch locally
print('✅ Gradio UI built — call demo.launch() to start')

In [ ]:
# ── Modal.com Deployment ──────────────────────────────────────────────────────
import modal

modal_app = modal.App('nigerian-flight-price-alert')

flight_image = modal.Image.debian_slim().pip_install([
    'openai>=1.0.0',
    'anthropic>=0.25.0',
    'google-generativeai>=0.7.0',
    'groq>=0.9.0',
    'requests>=2.31.0',
    'pydantic>=2.0.0',
    'python-dotenv>=1.0.0',
    'gradio>=4.0.0',
])

# Pre-configured routes for daily monitoring
MONITORED_ROUTES = [
    {'origin': 'LOS', 'destination': 'ABV', 'threshold': 40000},
    {'origin': 'LOS', 'destination': 'PHC', 'threshold': 45000},
    {'origin': 'ABV', 'destination': 'LOS', 'threshold': 40000},
]


@modal_app.function(
    image=flight_image,
    secrets=[modal.Secret.from_dotenv()],
    schedule=modal.Period(hours=24),
    timeout=300,
)
def daily_flight_scan() -> None:
    """
    Daily scheduled job (runs every 24 h on Modal) that scans pre-configured
    Nigerian routes and sends Pushover alerts when prices drop below thresholds.
    """
    from datetime import datetime, timedelta
    tomorrow = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')

    for cfg in MONITORED_ROUTES:
        origin, destination, threshold = cfg['origin'], cfg['destination'], cfg['threshold']
        print(f'🔍 Scanning {origin} → {destination} on {tomorrow}...')
        result   = search_flights(origin, destination, tomorrow)
        cheapest = result.cheapest_flight

        if cheapest.price_ngn <= threshold:
            src = '📡 LIVE' if cheapest.source == 'live' else '📊'
            notif = AlertNotification(
                title=f'✈️ Deal Alert: {origin}→{destination} {src}',
                message=(
                    f'{cheapest.airline} {cheapest.flight_number}\n'
                    f'💰 ₦{cheapest.price_ngn:,.0f} (threshold ₦{threshold:,.0f})\n'
                    f'🕐 Departs {cheapest.departure_time} on {tomorrow}'
                ),
                priority=1,
            )
            send_pushover_alert(notif)
            print(f'🔔 Alert sent — ₦{cheapest.price_ngn:,.0f}')
        else:
            print(f'💤 No deal — ₦{cheapest.price_ngn:,.0f} > ₦{threshold:,.0f}')


@modal_app.function(
    image=flight_image,
    secrets=[modal.Secret.from_dotenv()],
    allow_concurrent_inputs=10,
)
@modal.web_endpoint(label='flight-alert-ui')
def serve_gradio():
    """Serve the Gradio UI as a Modal ASGI web endpoint."""
    return gr.mount_gradio_app(app=demo, path='/')


# To deploy:
#   modal deploy week8/community_contributions/ijosh_week7_solution.ipynb
# To run daily scan locally:
#   modal run week8/community_contributions/ijosh_week7_solution.ipynb::daily_flight_scan
print('✅ Modal deployment configured')
print('   Deploy: modal deploy <this_file>')

In [ ]:
# ── Demo / Test Run (all outputs cleared for PR) ──────────────────────────────

print('=' * 60)
print('🛫 Nigerian Flight Price Alert — System Demo')
print('=' * 60)

# Test 1 — Pydantic models
print('\n📍 Test 1: Pydantic model validation')
test_flight = FlightOption(
    airline='Air Peace', flight_number='P4421',
    origin='LOS', destination='ABV',
    departure_time='09:00', arrival_time='10:05',
    price_ngn=42000.0,
)
assert test_flight.currency == 'NGN'
print(f'  ✅ FlightOption: {test_flight.airline} {test_flight.flight_number} @ ₦{test_flight.price_ngn:,.0f}')

notif = AlertNotification(title='Test Alert', message='Test message', priority=0)
assert -2 <= notif.priority <= 2
print(f'  ✅ AlertNotification: "{notif.title}"')

# Test 2 — Mock flight search (guaranteed to work without API keys)
print('\n📍 Test 2: Mock flight search (LOS → ABV)')
mock_options = _mock_search('LOS', 'ABV', '2026-03-15')
assert len(mock_options) > 0, 'Expected at least one mock flight'
assert all(28000 <= f.price_ngn <= 85000 for f in mock_options), 'Prices out of expected range'
print(f'  ✅ {len(mock_options)} flights generated')
print(f'  ✅ Price range: ₦{min(f.price_ngn for f in mock_options):,.0f} – ₦{max(f.price_ngn for f in mock_options):,.0f}')

# Test 3 — Full search_flights function
print('\n📍 Test 3: search_flights() end-to-end (LOS → PHC)')
result = search_flights('LOS', 'PHC', '2026-03-16')
assert isinstance(result, FlightSearchResult)
assert result.cheapest_flight.price_ngn <= result.all_options[-1].price_ngn
print(f'  ✅ Route: {result.route}')
print(f'  ✅ Options: {len(result.all_options)}')
print(f'  ✅ Cheapest: {result.cheapest_flight.airline} @ ₦{result.cheapest_flight.price_ngn:,.0f}')
print(f'  ✅ Source: {result.cheapest_flight.source}')

# Test 4 — AgentState
print('\n📍 Test 4: AgentState')
state = AgentState(origin='LOS', destination='KAN', travel_date='2026-03-17')
state.logs.append('Test log entry')
assert state.iteration == 0
assert len(state.logs) == 1
print(f'  ✅ AgentState: {state.origin} → {state.destination}')

print()
print('=' * 60)
print('✅ ALL TESTS PASSED')
print('=' * 60)
print()
print('📊 System Summary:')
print(f'  Routes supported  : {len(ROUTES)}')
print(f'  Nigerian airlines : {", ".join(a["name"] for a in NIGERIAN_AIRLINES)}')
print(f'  LLM ensemble size : {len(LLM_REGISTRY)}')
print(f'  Live data source  : Amadeus API (free sandbox)')
print(f'  Notifications     : Pushover')
print(f'  Deployment        : Modal.com (daily schedule)')
print()
print('🚀 Quick start:')
print('  state = orchestrator.run("LOS", "ABV", "2026-03-18")')
print('  demo.launch(server_port=7860)')